In [1]:
import os
import cv2
import tqdm
import numpy as np
import pandas as pd
import torchvision
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*video decoding and encoding capabilities of torchvision are deprecated.*",
    category=UserWarning,
)


In [ ]:
video_path = "/data/mtseng/TEyeD/raw_data/GazeinTheWild/VIDEOS/GW_6_1.mp4"
dataset_name = video_path.split("/")[-3]
annotation_name = os.path.basename(video_path)
print(f"Annotation name: {annotation_name}")

Annotation name: LPW_2_1.mp4


In [20]:
annotation_paths = {
    'validity': '../raw_data/{}/ANNOTATIONS/{}validity_pupil.txt'.format(dataset_name, annotation_name),
    'pupil_center': '../raw_data/{}/ANNOTATIONS/{}pupil_eli.txt'.format(dataset_name, annotation_name),
    'gaze_vector': '../raw_data/{}/ANNOTATIONS/{}gaze_vec.txt'.format(dataset_name, annotation_name),
    'iris_mask': '../raw_data/{}/ANNOTATIONS/{}iris_seg_2D.mp4'.format(dataset_name, annotation_name)
}

In [21]:
def load_annotations(annotation_paths):
    validity_path = annotation_paths['validity']
    validity_df = pd.read_csv(validity_path, sep=";", usecols=["FRAME", "VALIDITY"])

    gaze_path = annotation_paths['gaze_vector']
    gaze_df = pd.read_csv(gaze_path, sep=";", usecols=["FRAME", "X", "Y", "Z"])

    pupilCenter_path = annotation_paths['pupil_center']
    pupilCenter_df = pd.read_csv(pupilCenter_path, sep=";", usecols=["FRAME", "CENTER X", "CENTER Y"])

    df_all = (
        validity_df
        .merge(gaze_df, on="FRAME", how="left")
        .merge(pupilCenter_df, on="FRAME", how="left")
    )

    # rename columns
    df_all.rename(columns={
        "VALIDITY": "validity",
        "X": "gaze_x",
        "Y": "gaze_y",
        "Z": "gaze_z",
        "CENTER X": "pupil_center_x",
        "CENTER Y": "pupil_center_y",
    }, inplace=True)
    return df_all

In [22]:
annotation_df = load_annotations(annotation_paths)
print(annotation_df.head())
print(annotation_df.shape)

   FRAME  validity    gaze_x    gaze_y    gaze_z  pupil_center_x  \
0      1         1  0.386788 -0.354528  0.851296         426.053   
1      2         1  0.387256 -0.354021  0.851294         426.204   
2      3         1  0.387835 -0.353916  0.851074         426.391   
3      4         1  0.388518 -0.352340  0.851417         426.609   
4      5         1  0.390319 -0.351614  0.850893         427.185   

   pupil_center_y  
0         71.9242  
1         72.0781  
2         72.1051  
3         72.6013  
4         72.8265  
(2000, 7)


In [23]:
cap = cv2.VideoCapture(video_path)
iris_cap = cv2.VideoCapture(annotation_paths['iris_mask'])
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Frames per second: {fps}")
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
num_iris_frames = int(iris_cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Number of frames: {num_frames}")
assert num_frames == annotation_df.shape[0], "Number of frames in video and annotation do not match!"

chunksize = 30 * fps
out_video_dir = '../data/{}/{}/VIDEOS'.format(dataset_name, annotation_name.split(".")[0])
out_iris_dir = '../data/{}/{}/IRIS'.format(dataset_name, annotation_name.split(".")[0])
out_annotation_path = '../data/{}/{}/ANNOTATIONS'.format(dataset_name, annotation_name.split(".")[0])
os.makedirs(out_video_dir, exist_ok=True)
os.makedirs(out_iris_dir, exist_ok=True)
os.makedirs(out_annotation_path, exist_ok=True)

Frames per second: 95.0
Number of frames: 2000


In [24]:
video_frames = []
iris_frames = []
cap = cv2.VideoCapture(video_path)
H, W = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
target_H, target_W = 120, 160
assert H / target_H == W / target_W, "Aspect ratio of the video does not match the target aspect ratio!"
scaling_factor = H / target_H  # or W / target_W, they should be the same due to the assertion above
        
for _ in tqdm.tqdm(range(num_frames)):
    ret, frame = cap.read()
    ret_iris, iris_frame = iris_cap.read()
    if not ret_iris or not ret:
        break
    if dataset_name == "GazeinTheWild" and int(video_path[-5])%2 == 1:
        frame = cv2.flip(frame, 0)
    new_W = int(round(frame.shape[1] / scaling_factor))
    new_H = int(round(frame.shape[0] / scaling_factor))
    video_frames.append(cv2.resize(frame, (new_W, new_H)))
    iris_frames.append(cv2.resize(iris_frame, (new_W, new_H)))
cap.release()
iris_cap.release()
video_frames = np.array(video_frames)
iris_frames = np.array(iris_frames)

100%|██████████| 2000/2000 [00:04<00:00, 471.72it/s]


In [25]:
for i in tqdm.tqdm(range(0, num_frames, int(chunksize))):
    out_path = os.path.join(out_video_dir, f"{annotation_name.split('.')[0]}_chunk_{i//int(chunksize)}.mp4")
    if os.path.exists(out_path):
        print(f"Chunk {i//int(chunksize)} already exists, skipping...")
        continue

    video_chunk = video_frames[i:i+int(chunksize)]
    iris_chunk = iris_frames[i:i+int(chunksize)]

    out_iris_path = os.path.join(out_iris_dir, f"{annotation_name.split('.')[0]}_chunk_{i//int(chunksize)}.mp4")
    out_annotation_chunk_path = os.path.join(out_annotation_path, f"{annotation_name.split('.')[0]}_chunk_{i//int(chunksize)}.csv")
    annotation_chunk_df = annotation_df.iloc[i:i+int(chunksize)]
    annotation_chunk_df['pupil_center_x'] = (annotation_chunk_df['pupil_center_x'] / scaling_factor).round().astype(int)
    annotation_chunk_df['pupil_center_y'] = (annotation_chunk_df['pupil_center_y'] / scaling_factor).round().astype(int)
    annotation_chunk_df.to_csv(out_annotation_chunk_path, index=False)

    torchvision.io.write_video(out_path, video_chunk, fps)
    torchvision.io.write_video(out_iris_path, iris_chunk, fps)

100%|██████████| 1/1 [00:02<00:00,  2.78s/it]
